# 🎓 TSL Transformer Model Training on Google Colab

Train the Tanzania Sign Language (TSL) gesture recognition model using free GPU acceleration on Google Colab.

**Features:**
- ✅ GPU/CUDA acceleration (much faster than CPU)
- ✅ Google Drive integration for saving checkpoints
- ✅ Synthetic data generation for testing
- ✅ Early stopping and checkpoint management
- ✅ Real-time training progress monitoring

**Setup Time:** ~2 minutes | **Training Time:** ~1-2 minutes (with GPU)

## Step 1: Set Up Google Colab Environment

Configure the runtime and mount Google Drive for checkpoint storage.

In [ ]:
# Mount Google Drive for checkpoint storage
try:
    from google.colab import drive  # type: ignore[import-untyped]
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
    print("Your Google Drive is accessible at: /content/drive/MyDrive/")
except ImportError:
    print("Not running in Colab - skipping Drive mount.")
    print("Checkpoints will be saved locally instead.")


In [ ]:
# Check GPU availability
import torch

gpu_available = torch.cuda.is_available()
device = torch.device("cuda" if gpu_available else "cpu")

print(f"\n🎮 GPU Available: {gpu_available}")
if gpu_available:
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("   ⚠️  No GPU detected. Training will be slower on CPU.")

print(f"   Using device: {device}")


## Step 2: Install Required Libraries

Install PyTorch, MediaPipe, and other dependencies.

In [ ]:
import subprocess
import sys

# Install required packages quietly
packages = [
    "torch>=2.2.0",
    "mediapipe>=0.10.0",
    "numpy",
    "scikit-learn",
    "tqdm"
]

print("⏳ Installing dependencies...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All dependencies installed successfully!")

# Verify installations
import torch
import numpy as np
import mediapipe as mp
from tqdm import tqdm

print(f"\n📦 Versions:")
print(f"   PyTorch: {torch.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   MediaPipe: {mp.__version__}")


## Step 3: Load and Prepare Data

Generate synthetic hand landmark data for training. This simulates real gesture sequences with 21 hand landmarks (3D coordinates).

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Configuration
NUM_CLASSES = 12
SEQ_LENGTH = 20  # Sequence length (20 frames)
FEATURE_DIM = 126  # 2 hands × 21 landmarks × 3 coordinates

# Gesture class names
GESTURE_NAMES = [
    "Hello", "Stop", "One", "Peace", "Three", "Four",
    "Good", "I Love You", "Promise", "L-Shape", "Rock On", "Unknown"
]

print(f"📊 Dataset Configuration:")
print(f"   Number of gesture classes: {NUM_CLASSES}")
print(f"   Sequence length: {SEQ_LENGTH} frames")
print(f"   Feature dimension: {FEATURE_DIM} (2 hands × 21 landmarks × 3 coords)")
print(f"\n🎯 Gesture Classes:")
for i, name in enumerate(GESTURE_NAMES):
    print(f"   {i:2d}: {name}")


In [ ]:
def generate_synthetic_dataset(num_samples=1000, seq_length=SEQ_LENGTH, feature_dim=FEATURE_DIM):
    """
    Generate synthetic hand landmark sequences for each gesture class.
    
    Args:
        num_samples: Total number of samples to generate
        seq_length: Length of each sequence
        feature_dim: Dimensionality of hand landmarks (126 for 2 hands × 21 landmarks × 3)
    
    Returns:
        X: Array of shape (num_samples, seq_length, feature_dim)
        y: Array of shape (num_samples,) with class labels
    """
    X = []
    y = []
    samples_per_class = num_samples // NUM_CLASSES
    
    print(f"🔄 Generating {num_samples} synthetic samples ({samples_per_class} per class)...")
    
    for class_idx in range(NUM_CLASSES):
        for _ in range(samples_per_class):
            # Generate random hand landmarks
            sequence = np.random.randn(seq_length, feature_dim).astype(np.float32) * 0.5
            
            # Add class-specific patterns for variation
            sequence[:, :10] += (class_idx + 1) * 0.2
            sequence[:, 10:20] *= (class_idx + 1) * 0.1
            
            X.append(sequence)
            y.append(class_idx)
    
    return np.array(X), np.array(y)

# Generate datasets
print("\n📁 Creating datasets...")
X_train, y_train = generate_synthetic_dataset(num_samples=800, seq_length=SEQ_LENGTH)
X_val, y_val = generate_synthetic_dataset(num_samples=200, seq_length=SEQ_LENGTH)

print(f"\n✅ Datasets created:")
print(f"   Training:   {X_train.shape} (sequences, frames, features)")
print(f"   Validation: {X_val.shape}")
print(f"   Class distribution: {np.bincount(y_train)}")


In [ ]:
class TSLDataset(Dataset):
    """PyTorch Dataset for hand landmark sequences."""
    
    def __init__(self, sequences, labels):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# Create PyTorch datasets and dataloaders
BATCH_SIZE = 32

train_dataset = TSLDataset(X_train, y_train)
val_dataset = TSLDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✅ DataLoaders created:")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Batch size: {BATCH_SIZE}")


## Step 4: Build the Model

Define the Transformer architecture for gesture classification.

**Architecture:**
- Input: 126D hand landmark vectors (2 hands × 21 landmarks × 3 coordinates)
- Processing: Positional encoding + Multi-head self-attention (4 heads, 4 layers)
- Output: Classification logits for 12 gesture classes

In [ ]:
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

class TSLTransformer(nn.Module):
    """Transformer model for sign language gesture classification."""
    
    def __init__(self, num_classes=NUM_CLASSES, d_model=128, nhead=4, 
                 num_encoder_layers=4, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Input projection: 126 features → d_model dimensions
        self.input_projection = nn.Linear(FEATURE_DIM, d_model)
        
        # Positional encoding (learnable)
        self.pos_encoder = nn.Parameter(torch.randn(1, SEQ_LENGTH, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        
        # Classification head
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, 126) - hand landmark sequences
        Returns:
            logits: (batch_size, num_classes)
        """
        # Project to d_model dimensions
        x = self.input_projection(x)  # (batch, seq, d_model)
        
        # Add positional encoding
        x = x + self.pos_encoder[:, :x.size(1), :]
        
        # Transformer encoding
        x = self.encoder(x)  # (batch, seq, d_model)
        
        # Global average pooling
        x = x.transpose(1, 2)  # (batch, d_model, seq)
        x = self.pool(x).squeeze(-1)  # (batch, d_model)
        
        # Classification
        logits = self.fc(x)  # (batch, num_classes)
        return logits

# Initialize model
model = TSLTransformer(
    num_classes=NUM_CLASSES,
    d_model=128,
    nhead=4,
    num_encoder_layers=4,
    dim_feedforward=256
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ Model Architecture:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"\n📋 Model Summary:")
print(model)


## Step 5: Train the Model

Train with early stopping and checkpoint saving to Google Drive.

In [ ]:
import os
from pathlib import Path

# Setup training configuration
EPOCHS = 50
LR = 1e-3
EARLY_STOPPING_PATIENCE = 5

# Create checkpoint directory in Google Drive
checkpoint_dir = "/content/drive/MyDrive/TSL-Checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "tsl_transformer.pt")

print(f"🎯 Training Configuration:")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LR}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Checkpoint Directory: {checkpoint_dir}")


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for sequences, labels in loader:
        sequences, labels = sequences.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Metrics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

def validate(model, loader, criterion, device):
    """Validate model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for sequences, labels in loader:
            sequences, labels = sequences.to(device), labels.to(device)
            
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

print("✅ Training functions ready!")


In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

for epoch in range(EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Scheduler step
    scheduler.step()
    
    # Store history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    
    # Log every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}")
    
    # Early stopping with checkpoint saving
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        
        # Save checkpoint
        torch.save(model.state_dict(), checkpoint_path)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"            💾 Checkpoint saved! (Best so far)")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n⏹️  Early stopping at epoch {epoch+1} (best: epoch {best_epoch+1})")
            print(f"    Best validation loss: {best_val_loss:.4f}")
            break

print("="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)


## Step 6: Evaluate Model Performance

Evaluate the trained model on test data and visualize results.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best checkpoint
model.load_state_dict(torch.load(checkpoint_path))

# Evaluate on validation set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for sequences, labels in val_loader:
        sequences = sequences.to(device)
        outputs = model(sequences)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print("📊 MODEL EVALUATION METRICS")
print("="*50)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print("\n📋 CLASSIFICATION REPORT")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=GESTURE_NAMES, zero_division=0))


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Training Loss', marker='o', markersize=3)
axes[0].plot(history['val_loss'], label='Validation Loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Training Accuracy', marker='o', markersize=3)
axes[1].plot(history['val_acc'], label='Validation Accuracy', marker='s', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, 'training_history.png'), dpi=100, bbox_inches='tight')
plt.show()

print("✅ Training history plot saved!")


In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=GESTURE_NAMES, yticklabels=GESTURE_NAMES,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Gesture Classification')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, 'confusion_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix plot saved!")


In [ ]:
# Create a summary file
summary_text = f"""
TSL TRANSFORMER MODEL - TRAINING SUMMARY
{'='*60}

Training Configuration:
  - Epochs: {EPOCHS}
  - Batch Size: {BATCH_SIZE}
  - Learning Rate: {LR}
  - Sequence Length: {SEQ_LENGTH}
  - Feature Dimension: {FEATURE_DIM}

Model Architecture:
  - Total Parameters: {total_params:,}
  - Trainable Parameters: {trainable_params:,}
  - Encoder Layers: 4
  - Attention Heads: 4
  - Model Dimension: 128

Training Results:
  - Best Epoch: {best_epoch + 1}
  - Best Validation Loss: {best_val_loss:.4f}
  - Final Validation Accuracy: {history['val_acc'][best_epoch]:.4f}

Gesture Classes (12 total):
{chr(10).join([f"  {i:2d}. {name}" for i, name in enumerate(GESTURE_NAMES)])}

Files Generated:
  - tsl_transformer.pt (model checkpoint)
  - training_history.png (training curves)
  - confusion_matrix.png (prediction analysis)

{'='*60}
Training completed successfully! ✅
"""

summary_path = os.path.join(checkpoint_dir, 'training_summary.txt')
with open(summary_path, 'w') as f:
    f.write(summary_text)

print(summary_text)
print(f"\n📄 Summary saved to: {summary_path}")


## Step 7: Save the Trained Model

The model checkpoint has been automatically saved to Google Drive during training. Here's how to use it locally.

In [ ]:
# Verify checkpoint was saved
import os

if os.path.exists(checkpoint_path):
    checkpoint_size = os.path.getsize(checkpoint_path) / (1024**2)  # Convert to MB
    print("✅ CHECKPOINT SAVED TO GOOGLE DRIVE")
    print("="*60)
    print(f"📁 Location: {checkpoint_path}")
    print(f"💾 Size: {checkpoint_size:.2f} MB")
    print(f"✓ Model trained for {best_epoch + 1} epochs")
    print(f"✓ Best validation loss: {best_val_loss:.4f}")
else:
    print("❌ Checkpoint file not found!")


### Download Instructions

1. **Open Google Drive:** https://drive.google.com/drive/my-drive
2. **Find the checkpoint** in the `TSL-Checkpoints` folder
3. **Download** `tsl_transformer.pt`
4. **Place it locally** in your project: `backend/checkpoints/tsl_transformer.pt`
5. **Restart backend server** to use the new model

### What You'll Get

- **Model Checkpoint:** `tsl_transformer.pt` (contains all trained weights)
- **Training History:** `training_history.png` (loss and accuracy curves)
- **Confusion Matrix:** `confusion_matrix.png` (prediction analysis)

### Next Steps

After downloading the checkpoint:
1. Start your backend server: `cd backend && python main.py`
2. Start your frontend: `cd frontend && npm run dev`
3. Click "Start Detection" to use your trained gesture recognition model!

## 🚀 Quick Start Guide

### To use this notebook:

1. **Copy the notebook URL:** Get the link from your browser
2. **Open in Colab:** Go to https://colab.research.google.com and open from GitHub (or upload the .ipynb file)
3. **Run all cells:** Click "Runtime" → "Run all"
4. **Wait for training:** Takes ~1-2 minutes on GPU
5. **Download checkpoint:** From Google Drive → TSL-Checkpoints folder

### Pro Tips:

- 🎯 **Use GPU:** The notebook automatically detects and uses GPU if available
- 💾 **Auto-save:** Checkpoint saves to Google Drive during training
- ⏹️ **Early stopping:** Training stops automatically if no improvement for 5 epochs
- 📊 **Visualizations:** Training curves and confusion matrix generated automatically

### Troubleshooting:

| Issue | Solution |
|-------|----------|
| "Google Drive not mounted" | Run Step 1 first to mount your Drive |
| GPU not available | Colab may have no GPU quota. Re-run later or use CPU (slower) |
| Checkpoint not saving | Check if `/MyDrive` folder exists in Google Drive |
| ModuleNotFoundError | Make sure dependencies installed in Step 2 |

---

**Questions?** Check the [GitHub Issues](https://github.com/your-username/TSL-System/issues) or re-run the notebook.